# Assignment 2 DSC 102 FA25

## Introduction

In this assignment we will conduct data engineering for the Amazon dataset. It is divided into 2 parts. The extracted features in Part 1 will be used for the Part 2 of assignment, where you train a model (or models) to predict user ratings for a product.

We will be using Apache Spark for this assignment. The default Spark API will be DataFrame, as it is now the recommended choice over the RDD API. That being said, please feel free to switch back to the RDD API if you see it as a better fit for the task. We provide you an option to request RDD format to start with. Also you can switch between DataFrame and RDD in your solution. 

Another newer API is Koalas, which is also avaliable. However, it has constraints and is not applicable to most tasks. Refer to the PA statement for detail.

### Set the following parameters

In [1]:
PID = 'a69043483' # your pid, for instance: 'a43223333'
INPUT_FORMAT = 'dataframe' # choose a format of your input data, valid options: 'dataframe', 'rdd', 'koalas'

In [2]:
# Boiler plates, do NOT modify
%load_ext autoreload
%autoreload 2
import os
import getpass
from pyspark.sql import SparkSession
from utilities import SEED
from utilities import PA2Test
from utilities import PA2Data
from utilities import data_cat
from pa2_main import PA2Executor
import time
if INPUT_FORMAT == 'dataframe':
    import pyspark.ml as M
    import pyspark.sql.functions as F
    import pyspark.sql.types as T
if INPUT_FORMAT == 'koalas':
    import databricks.koalas as ks
elif INPUT_FORMAT == 'rdd':
    import pyspark.mllib as M
    from pyspark.mllib.feature import Word2Vec
    from pyspark.mllib.linalg import Vectors
    from pyspark.mllib.linalg.distributed import RowMatrix

os.environ['PYSPARK_SUBMIT_ARGS'] = '--py-files utilities.py,assignment2.py \
--deploy-mode client \
pyspark-shell'

class args:
    review_filename = data_cat.review_filename
    product_filename = data_cat.product_filename
    product_processed_filename = data_cat.product_processed_filename
    ml_features_train_filename = data_cat.ml_features_train_filename
    ml_features_test_filename = data_cat.ml_features_test_filename
    output_root = '/home/{}/{}-pa2/test_results'.format(getpass.getuser(), PID)
    test_results_root = data_cat.test_results_root
    pid = PID

pa2 = PA2Executor(args, input_format=INPUT_FORMAT)
data_io = pa2.data_io
data_dict = pa2.data_dict
begin = time.time()


26/05/18 18:27:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/bitnami/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Loading datasets ...Done


In [33]:
# Import your own dependencies
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

#-----------------------------

# Part 1: Feature Engineering

In [3]:
# Bring the part_1 datasets to memory and de-cache part_2 datasets. 
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_1')

# Task0: warm up 
This task is provided for you to get familiar with Spark API. We will use the dataframe API to demonstrate. Solution is given to you and this task won't be graded.

Refer to https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html for API guide.

The task is to implement the function below. Given the ```product_data``` table:
1. Take and print five rows.

1. Select only the ```asin``` column, then print five rows of it.

1. Select the row where ```asin = B00I8KEOTM``` and print it.

1. Count the total number of rows.

1. Calculate the mean ```price```.

1. You need to conduct the above operations, then extract some statistics out of the generated columns. You need to put the statistics in a python dictionary named ```res```. The description and schema of it are as follows:
    ```
    res
     | -- count_total: int -- count of total rows of the entire table after your operations
     | -- mean_price: float -- mean value of column price
    ```

In [4]:
def task_0(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    product_data.show(5)
    product_data[['asin']].show(5)
    product_data.where(F.col('asin') == 'B00I8KEOTM').show()
    count_rows = product_data.count()
    mean_price = product_data.select(F.avg(F.col('price'))).head()[0]
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmatically. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {'count_total': None, 'mean_price': None}
    
    # Modify res:

    res['count_total'] = count_rows
    res['mean_price'] = mean_price

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    return res
    # -------------------------------------------------------------------------

In [5]:
if INPUT_FORMAT == 'dataframe':
    res = task_0(data_io, data_dict['product'])
    pa2.tests.test(res, 'task_0')

+----------+--------------------+--------------------+--------------------+-----+--------------------+
|      asin|           salesRank|          categories|               title|price|             related|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
|B00I8HVV6E|{Home &amp; Kitch...|[[Home & Kitchen,...|Intelligent Desig...|27.99|{also_viewed -> [...|
|B00I8KEOTM|                null|[[Apps for Androi...|                null| null|{also_viewed -> [...|
|B00I8KCW4G|{Clothing -> 2233...|[[Clothing, Shoes...|eShakti Women's P...|41.95|{also_viewed -> [...|
|B00I8JKCQW|{Clothing -> 1405...|[[Clothing, Shoes...|Lady Slimming Mid...| null|{also_viewed -> [...|
|B00I8JKI8E|{Home &amp; Kitch...|[[Clothing, Shoes...|3 Tier Bangle Bra...|24.99|{also_viewed -> [...|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
only showing top 5 rows

+----------+
|      asin|
+----------+
|B00I8HVV

tests for task_0 --------------------------------------------------------------
2/2 passed
-------------------------------------------------------------------------------


In [6]:
def _as_float(x):
    return None if x is None else float(x)

def _as_int(x):
    return 0 if x is None else int(x)

def _null_count(col_name):
    return F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0))

# Task1

In [9]:
# %load -s task_1 assignment2.py
def task_1(data_io, review_data, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    ratings = (
        review_data
        .groupBy(asin_column)
        .agg(
            F.avg(F.col(overall_column)).alias(mean_rating_column),
            F.count(F.col(overall_column)).alias(count_rating_column)
        )
    )

    transformed = product_data.join(ratings, on=asin_column, how='left')
    
    stats = transformed.agg(
        F.count("*").alias("count_total"),
        F.avg(mean_rating_column).alias("mean_meanRating"),
        F.variance(mean_rating_column).alias("variance_meanRating"),
        _null_count(mean_rating_column).alias("numNulls_meanRating"),
        F.avg(count_rating_column).alias("mean_countRating"),
        F.variance(count_rating_column).alias("variance_countRating"),
        _null_count(count_rating_column).alias("numNulls_countRating")
    ).first()

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmaticly. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {
        'count_total': None,
        'mean_meanRating': None,
        'variance_meanRating': None,
        'numNulls_meanRating': None,
        'mean_countRating': None,
        'variance_countRating': None,
        'numNulls_countRating': None
    }
    # Modify res:
    res = {
        'count_total': _as_int(stats["count_total"]),
        'mean_meanRating': _as_float(stats["mean_meanRating"]),
        'variance_meanRating': _as_float(stats["variance_meanRating"]),
        'numNulls_meanRating': _as_int(stats["numNulls_meanRating"]),
        'mean_countRating': _as_float(stats["mean_countRating"]),
        'variance_countRating': _as_float(stats["variance_countRating"]),
        'numNulls_countRating': _as_int(stats["numNulls_countRating"])
    }


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_1')
    return res
    # -------------------------------------------------------------------------


In [10]:
res = task_1(data_io, data_dict['review'], data_dict['product'])
pa2.tests.test(res, 'task_1')

tests for task_1 --------------------------------------------------------------
Test 1/7 : count_total ... Pass
Test 2/7 : mean_countRating ... Pass
Test 3/7 : mean_meanRating ... Pass
Test 4/7 : numNulls_countRating ... Pass
Test 5/7 : numNulls_meanRating ... Pass
Test 6/7 : variance_countRating ... Pass
Test 7/7 : variance_meanRating ... Pass
7/7 passed
-------------------------------------------------------------------------------


True


# Task 2

In [13]:
# %load -s task_2 assignment2.py
def task_2(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    salesRank_column = 'salesRank'
    categories_column = 'categories'
    asin_column = 'asin'
    # Outputs:
    category_column = 'category'
    bestSalesCategory_column = 'bestSalesCategory'
    bestSalesRank_column = 'bestSalesRank'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    raw_category = F.col(categories_column).getItem(0).getItem(0)

    transformed = (
        product_data
        .withColumn(
            category_column,
            F.when(
                (F.col(categories_column).isNotNull()) &
                (F.size(F.col(categories_column)) > 0) &
                (F.col(categories_column).getItem(0).isNotNull()) &
                (F.size(F.col(categories_column).getItem(0)) > 0) &
                (raw_category.isNotNull()) &
                (F.trim(raw_category) != ""),
                raw_category
            ).otherwise(F.lit(None).cast(T.StringType()))
        )
        .withColumn(
            bestSalesCategory_column,
            F.when(
                (F.col(salesRank_column).isNotNull()) &
                (F.size(F.col(salesRank_column)) > 0),
                F.map_keys(F.col(salesRank_column)).getItem(0)
            ).otherwise(F.lit(None).cast(T.StringType()))
        )
        .withColumn(
            bestSalesRank_column,
            F.when(
                (F.col(salesRank_column).isNotNull()) &
                (F.size(F.col(salesRank_column)) > 0),
                F.map_values(F.col(salesRank_column)).getItem(0)
            ).otherwise(F.lit(None).cast(T.IntegerType()))
        )
    )

    stats = transformed.agg(
        F.count("*").alias("count_total"),
        F.avg(bestSalesRank_column).alias("mean_bestSalesRank"),
        F.variance(bestSalesRank_column).alias("variance_bestSalesRank"),
        _null_count(category_column).alias("numNulls_category"),
        F.countDistinct(category_column).alias("countDistinct_category"),
        _null_count(bestSalesCategory_column).alias("numNulls_bestSalesCategory"),
        F.countDistinct(bestSalesCategory_column).alias("countDistinct_bestSalesCategory")
    ).first()


    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'mean_bestSalesRank': None,
        'variance_bestSalesRank': None,
        'numNulls_category': None,
        'countDistinct_category': None,
        'numNulls_bestSalesCategory': None,
        'countDistinct_bestSalesCategory': None
    }
    # Modify res:

    res = {
        'count_total': _as_int(stats["count_total"]),
        'mean_bestSalesRank': _as_float(stats["mean_bestSalesRank"]),
        'variance_bestSalesRank': _as_float(stats["variance_bestSalesRank"]),
        'numNulls_category': _as_int(stats["numNulls_category"]),
        'countDistinct_category': _as_int(stats["countDistinct_category"]),
        'numNulls_bestSalesCategory': _as_int(stats["numNulls_bestSalesCategory"]),
        'countDistinct_bestSalesCategory': _as_int(stats["countDistinct_bestSalesCategory"])
    }


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_2')
    return res
    # -------------------------------------------------------------------------


In [14]:
res = task_2(data_io, data_dict['product'])
pa2.tests.test(res, 'task_2')

tests for task_2 --------------------------------------------------------------
Test 1/7 : countDistinct_bestSalesCategory ... Pass
Test 2/7 : countDistinct_category ... Pass
Test 3/7 : count_total ... Pass
Test 4/7 : mean_bestSalesRank ... Pass
Test 5/7 : numNulls_bestSalesCategory ... Pass
Test 6/7 : numNulls_category ... Pass
Test 7/7 : variance_bestSalesRank ... Pass
7/7 passed
-------------------------------------------------------------------------------


True

# Task 3





In [15]:
# %load -s task_3 assignment2.py
def task_3(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    price_column = 'price'
    attribute = 'also_viewed'
    related_column = 'related'
    # Outputs:
    meanPriceAlsoViewed_column = 'meanPriceAlsoViewed'
    countAlsoViewed_column = 'countAlsoViewed'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    rid_column = "__rid"
    also_viewed_column = "__also_viewed"
    viewed_asin_column = "__viewed_asin"
    viewed_price_column = "__viewed_price"

    base = (
        product_data
        .withColumn(rid_column, F.monotonically_increasing_id())
        .withColumn(also_viewed_column, F.col(related_column).getItem(attribute))
        .withColumn(
            countAlsoViewed_column,
            F.when(
                (F.col(also_viewed_column).isNotNull()) &
                (F.size(F.col(also_viewed_column)) > 0),
                F.size(F.col(also_viewed_column))
            ).otherwise(F.lit(None).cast(T.IntegerType()))
        )
    )

    exploded = (
        base
        .where(
            (F.col(also_viewed_column).isNotNull()) &
            (F.size(F.col(also_viewed_column)) > 0)
        )
        .select(
            rid_column,
            F.explode(F.col(also_viewed_column)).alias(viewed_asin_column)
        )
    )

    prices = (
        product_data
        .select(
            F.col(asin_column).alias(viewed_asin_column),
            F.col(price_column).alias(viewed_price_column)
        )
        .where(F.col(viewed_price_column).isNotNull())
    )

    mean_prices = (
        exploded
        .join(F.broadcast(prices), on=viewed_asin_column, how='left')
        .groupBy(rid_column)
        .agg(F.avg(viewed_price_column).alias(meanPriceAlsoViewed_column))
    )

    transformed = base.join(mean_prices, on=rid_column, how='left')
    
    stats = transformed.agg(
        F.count("*").alias("count_total"),
        F.avg(meanPriceAlsoViewed_column).alias("mean_meanPriceAlsoViewed"),
        F.variance(meanPriceAlsoViewed_column).alias("variance_meanPriceAlsoViewed"),
        _null_count(meanPriceAlsoViewed_column).alias("numNulls_meanPriceAlsoViewed"),
        F.avg(countAlsoViewed_column).alias("mean_countAlsoViewed"),
        F.variance(countAlsoViewed_column).alias("variance_countAlsoViewed"),
        _null_count(countAlsoViewed_column).alias("numNulls_countAlsoViewed")
    ).first()

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'mean_meanPriceAlsoViewed': None,
        'variance_meanPriceAlsoViewed': None,
        'numNulls_meanPriceAlsoViewed': None,
        'mean_countAlsoViewed': None,
        'variance_countAlsoViewed': None,
        'numNulls_countAlsoViewed': None
    }
    # Modify res:
    res = {
        'count_total': _as_int(stats["count_total"]),
        'mean_meanPriceAlsoViewed': _as_float(stats["mean_meanPriceAlsoViewed"]),
        'variance_meanPriceAlsoViewed': _as_float(stats["variance_meanPriceAlsoViewed"]),
        'numNulls_meanPriceAlsoViewed': _as_int(stats["numNulls_meanPriceAlsoViewed"]),
        'mean_countAlsoViewed': _as_float(stats["mean_countAlsoViewed"]),
        'variance_countAlsoViewed': _as_float(stats["variance_countAlsoViewed"]),
        'numNulls_countAlsoViewed': _as_int(stats["numNulls_countAlsoViewed"])
    }


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_3')
    return res
    # -------------------------------------------------------------------------


In [16]:
res = task_3(data_io, data_dict['product'])
pa2.tests.test(res, 'task_3')

tests for task_3 --------------------------------------------------------------
Test 1/7 : count_total ... Pass
Test 2/7 : mean_countAlsoViewed ... Pass
Test 3/7 : mean_meanPriceAlsoViewed ... Pass
Test 4/7 : numNulls_countAlsoViewed ... Pass
Test 5/7 : numNulls_meanPriceAlsoViewed ... Pass
Test 6/7 : variance_countAlsoViewed ... Pass
Test 7/7 : variance_meanPriceAlsoViewed ... Pass
7/7 passed
-------------------------------------------------------------------------------


True

# Task 4

In [17]:
# %load -s task_4 assignment2.py
def task_4(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    price_column = 'price'
    title_column = 'title'
    # Outputs:
    meanImputedPrice_column = 'meanImputedPrice'
    medianImputedPrice_column = 'medianImputedPrice'
    unknownImputedTitle_column = 'unknownImputedTitle'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    price_float_column = "__price_float"

    price_df = product_data.withColumn(
        price_float_column,
        F.col(price_column).cast(T.FloatType())
    )
    
    mean_price = price_df.agg(F.avg(price_float_column).alias("mean_price")).first()["mean_price"]

    median_list = (
        price_df
        .where(F.col(price_float_column).isNotNull())
        .approxQuantile(price_float_column, [0.5], 0.0)
    )
    median_price = median_list[0] if len(median_list) > 0 else None
    
    transformed = (
        price_df
        .withColumn(
            meanImputedPrice_column,
            F.when(F.col(price_float_column).isNull(), F.lit(mean_price))
             .otherwise(F.col(price_float_column))
             .cast(T.FloatType())
        )
        .withColumn(
            medianImputedPrice_column,
            F.when(F.col(price_float_column).isNull(), F.lit(median_price))
             .otherwise(F.col(price_float_column))
             .cast(T.FloatType())
        )
        .withColumn(
            unknownImputedTitle_column,
            F.when(
                F.col(title_column).isNull() | (F.trim(F.col(title_column)) == ""),
                F.lit("unknown")
            ).otherwise(F.col(title_column))
        )
    )
    
    stats = transformed.agg(
        F.count("*").alias("count_total"),
        F.avg(meanImputedPrice_column).alias("mean_meanImputedPrice"),
        F.variance(meanImputedPrice_column).alias("variance_meanImputedPrice"),
        _null_count(meanImputedPrice_column).alias("numNulls_meanImputedPrice"),
        F.avg(medianImputedPrice_column).alias("mean_medianImputedPrice"),
        F.variance(medianImputedPrice_column).alias("variance_medianImputedPrice"),
        _null_count(medianImputedPrice_column).alias("numNulls_medianImputedPrice"),
        F.sum(
            F.when(F.col(unknownImputedTitle_column) == "unknown", 1).otherwise(0)
        ).alias("numUnknowns_unknownImputedTitle")
    ).first()


    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'mean_meanImputedPrice': None,
        'variance_meanImputedPrice': None,
        'numNulls_meanImputedPrice': None,
        'mean_medianImputedPrice': None,
        'variance_medianImputedPrice': None,
        'numNulls_medianImputedPrice': None,
        'numUnknowns_unknownImputedTitle': None
    }
    # Modify res:
    res = {
        'count_total': _as_int(stats["count_total"]),
        'mean_meanImputedPrice': _as_float(stats["mean_meanImputedPrice"]),
        'variance_meanImputedPrice': _as_float(stats["variance_meanImputedPrice"]),
        'numNulls_meanImputedPrice': _as_int(stats["numNulls_meanImputedPrice"]),
        'mean_medianImputedPrice': _as_float(stats["mean_medianImputedPrice"]),
        'variance_medianImputedPrice': _as_float(stats["variance_medianImputedPrice"]),
        'numNulls_medianImputedPrice': _as_int(stats["numNulls_medianImputedPrice"]),
        'numUnknowns_unknownImputedTitle': _as_int(stats["numUnknowns_unknownImputedTitle"])
    }


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_4')
    return res
    # -------------------------------------------------------------------------


In [18]:
res = task_4(data_io, data_dict['product'])
pa2.tests.test(res, 'task_4')

tests for task_4 --------------------------------------------------------------
Test 1/8 : count_total ... Pass
Test 2/8 : mean_meanImputedPrice ... Pass
Test 3/8 : mean_medianImputedPrice ... Pass
Test 4/8 : numNulls_meanImputedPrice ... Pass
Test 5/8 : numNulls_medianImputedPrice ... Pass
Test 6/8 : numUnknowns_unknownImputedTitle ... Pass
Test 7/8 : variance_meanImputedPrice ... Pass
Test 8/8 : variance_medianImputedPrice ... Pass
8/8 passed
-------------------------------------------------------------------------------


True

# Task 5

In [23]:
# %load -s task_5 assignment2.py
def task_5(data_io, product_processed_data, word_0, word_1, word_2):
    # -----------------------------Column names--------------------------------
    # Inputs:
    title_column = 'title'
    # Outputs:
    titleArray_column = 'titleArray'
    titleVector_column = 'titleVector'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    product_processed_data_output = product_processed_data.withColumn(
        titleArray_column,
        F.split(F.lower(F.col(title_column)), " ")
    )

    word2vec = M.feature.Word2Vec(
        minCount=100,
        vectorSize=16,
        seed=SEED,
        numPartitions=4,
        inputCol=titleArray_column,
        outputCol=titleVector_column
    )
    
    model = word2vec.fit(product_processed_data_output)

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'size_vocabulary': None,
        'word_0_synonyms': [(None, None), ],
        'word_1_synonyms': [(None, None), ],
        'word_2_synonyms': [(None, None), ]
    }
    # Modify res:
    res['count_total'] = product_processed_data_output.count()
    res['size_vocabulary'] = model.getVectors().count()
    for name, word in zip(
        ['word_0_synonyms', 'word_1_synonyms', 'word_2_synonyms'],
        [word_0, word_1, word_2]
    ):
        res[name] = model.findSynonymsArray(word, 10)
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_5')
    return res
    # -------------------------------------------------------------------------


In [24]:
res = task_5(data_io, data_dict['product_processed'], 'piano', 'rice', 'laptop')
pa2.tests.test(res, 'task_5')

26/05/18 19:04:56 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/18 19:04:56 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS


tests for task_5 --------------------------------------------------------------
Test 1/8 : count_total ... Pass
Test 2/8 : size_vocabulary ... Pass
Test 3/8 : word_0_synonyms-length ... Pass
Test 4/8 : word_0_synonyms-correctness ... Pass
Test 5/8 : word_1_synonyms-length ... Pass
Test 6/8 : word_1_synonyms-correctness ... Pass
Test 7/8 : word_2_synonyms-length ... Pass
Test 8/8 : word_2_synonyms-correctness ... Pass
5/5 passed
-------------------------------------------------------------------------------


True

# Task 6

In [27]:
# %load -s task_6 assignment2.py
def task_6(data_io, product_processed_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    category_column = 'category'
    # Outputs:
    categoryIndex_column = 'categoryIndex'
    categoryOneHot_column = 'categoryOneHot'
    categoryPCA_column = 'categoryPCA'
    # -------------------------------------------------------------------------    

    # ---------------------- Your implementation begins------------------------
    indexer = M.feature.StringIndexer(
        inputCol=category_column,
        outputCol=categoryIndex_column
    )

    indexed = indexer.fit(product_processed_data).transform(product_processed_data)

    encoder = M.feature.OneHotEncoder(
            inputCols=[categoryIndex_column],
            outputCols=[categoryOneHot_column],
            dropLast=False
        )
    encoded = encoder.fit(indexed).transform(indexed)

    pca = M.feature.PCA(
        k=15,
        inputCol=categoryOneHot_column,
        outputCol=categoryPCA_column
    )

    transformed = pca.fit(encoded).transform(encoded)

    mean_onehot = (
        transformed
        .select(M.stat.Summarizer.mean(F.col(categoryOneHot_column)).alias("mean"))
        .first()["mean"]
    )

    mean_pca = (
        transformed
        .select(M.stat.Summarizer.mean(F.col(categoryPCA_column)).alias("mean"))
        .first()["mean"]
    )

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'meanVector_categoryOneHot': [None, ],
        'meanVector_categoryPCA': [None, ]
    }
    # Modify res:
    res = {
        'count_total': int(transformed.count()),
        'meanVector_categoryOneHot': [float(x) for x in mean_onehot.toArray()],
        'meanVector_categoryPCA': [float(x) for x in mean_pca.toArray()]
    }

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_6')
    return res
    # -------------------------------------------------------------------------


In [28]:
res = task_6(data_io, data_dict['product_processed'])
pa2.tests.test(res, 'task_6')

26/05/18 19:06:42 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
26/05/18 19:06:42 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK


tests for task_6 --------------------------------------------------------------
Test 1/9 : count_total ... Pass
Test 2/9 : meanVector_categoryOneHot-length ... Pass
Test 3/9 : meanVector_categoryOneHot-sum ... Pass
Test 4/9 : meanVector_categoryOneHot-mean ... Pass
Test 5/9 : meanVector_categoryOneHot-variance ... Pass
Test 6/9 : meanVector_categoryPCA-length ... Pass
Test 7/9 : meanVector_categoryPCA-sum ... Pass
Test 8/9 : meanVector_categoryPCA-mean ... Pass
Test 9/9 : meanVector_categoryPCA-variance ... Pass
9/9 passed
-------------------------------------------------------------------------------


True

In [29]:
print ("End to end time: {}".format(time.time()-begin))

End to end time: 2293.7489309310913


# Part 2: Model Selection

In [30]:
# Bring the part_2 datasets to memory and de-cache part_1 datasets.
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_2')

# Task 7

In [34]:
def task_7(data_io, train_data, test_data):
    
    # ---------------------- Your implementation begins------------------------
    dt = DecisionTreeRegressor(
        featuresCol="features",
        labelCol="overall",
        maxDepth=5
    )

    model = dt.fit(train_data)
    predictions = model.transform(test_data)

    evaluator = RegressionEvaluator(
        labelCol="overall",
        predictionCol="prediction",
        metricName="rmse"
    )

    test_rmse = evaluator.evaluate(predictions)    
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': None
    }
    # Modify res:
    res = {
        'test_rmse': float(test_rmse)
    }

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_7')
    return res
    # -------------------------------------------------------------------------

In [35]:
res = task_7(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_7')

tests for task_7 --------------------------------------------------------------
Test 1/1 : test_rmse ... Pass
1/1 passed
-------------------------------------------------------------------------------


True

# Task 8

In [36]:
def task_8(data_io, train_data, test_data):
    
    # ---------------------- Your implementation begins------------------------
    train_split, valid_split = train_data.randomSplit([0.75, 0.25], seed=SEED)

    evaluator = RegressionEvaluator(
        labelCol="overall",
        predictionCol="prediction",
        metricName="rmse"
    )

    depths = [5, 7, 9, 12]
    models = {}
    valid_rmses = {}

    for depth in depths:
        dt = DecisionTreeRegressor(
            featuresCol="features",
            labelCol="overall",
            maxDepth=depth
        )

        model = dt.fit(train_split)
        models[depth] = model

        valid_predictions = model.transform(valid_split)
        valid_rmses[depth] = float(evaluator.evaluate(valid_predictions))

    best_depth = min(valid_rmses, key=valid_rmses.get)
    best_model = models[best_depth]

    test_predictions = best_model.transform(test_data)
    test_rmse = float(evaluator.evaluate(test_predictions))
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': None,
        'valid_rmse_depth_5': None,
        'valid_rmse_depth_7': None,
        'valid_rmse_depth_9': None,
        'valid_rmse_depth_12': None,
    }
    # Modify res:
    res = {
        'test_rmse': test_rmse,
        'valid_rmse_depth_5': valid_rmses[5],
        'valid_rmse_depth_7': valid_rmses[7],
        'valid_rmse_depth_9': valid_rmses[9],
        'valid_rmse_depth_12': valid_rmses[12],
    }

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_8')
    return res
    # -------------------------------------------------------------------------

In [37]:
res = task_8(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_8')

tests for task_8 --------------------------------------------------------------
Test 1/5 : test_rmse ... Pass
Test 2/5 : valid_rmse_depth_12 ... Pass
Test 3/5 : valid_rmse_depth_5 ... Pass
Test 4/5 : valid_rmse_depth_7 ... Pass
Test 5/5 : valid_rmse_depth_9 ... Pass
5/5 passed
-------------------------------------------------------------------------------


True

In [38]:
print ("End to end time: {}".format(time.time()-begin))

End to end time: 3522.1762988567352
